In [ ]:
!pip install azure-eventhub

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import StringType

# import Even

In [ ]:


# Event Hubs configuration
connectionString = "Endpoint=sb://prateek.servicebus.windows.net/;SharedAccessKeyName=prateekstream_prateekevent_policy;SharedAccessKey=ANBwWxwt0eyAv7wcci+g/500bLtHzS6tj+AEhEbGZ/8=;EntityPath=prateekevent"
eventHubName = "prateekevent"

ehConf = {
    "eventhubs.connectionString": sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(connectionString),
    "eventhubs.consumerGroup": "$Default"  # Change if using a different consumer group
}


In [ ]:
df = spark.readStream.format("eventhubs").options(**ehConf).load()

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
 
transaction_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("account_id", StringType(), True),  # Updated from "customer_id" to "account_id"
    StructField("transaction_date", StringType(), True),
    StructField("transaction_type", StringType(), True),  # Updated from "product_id" to "transaction_type"
    StructField("transaction_amount", IntegerType(), True),  # Updated from DoubleType to IntegerType
    StructField("merchant_id", StringType(), True)  # Updated from "payment_type" to "merchant_id"
])

df = df.selectExpr("CAST(body AS STRING) as json_data")
df = df.withColumn("data", from_json(col("json_data"), transaction_schema))
df = df.select("data.*")  # Extract only required fields

In [ ]:

# COMMAND ----------

display(df)

transaction_id,account_id,transaction_date,transaction_type,transaction_amount,merchant_id,year_month
RT_TXN20000,ACC1088,2025-02-01 09:19:36,Subscription Payment,15000,MER2010,2025-02
RT_TXN20001,ACC1074,2025-02-04 12:47:36,ATM Withdrawal,10000,MER2093,2025-02
RT_TXN20002,ACC1080,2025-02-18 09:23:36,Online Purchase,2000,MER2052,2025-02
RT_TXN20003,ACC1082,2025-01-30 06:07:36,Deposit,20000,MER2036,2025-01
RT_TXN20004,ACC1011,2025-01-30 18:34:36,Subscription Payment,200000,MER2060,2025-01
RT_TXN20005,ACC1005,2025-02-01 18:25:36,Withdrawal,15000,MER2029,2025-02
RT_TXN20006,ACC1030,2025-02-09 13:14:36,ATM Withdrawal,10000,MER2001,2025-02
RT_TXN20007,ACC1022,2025-02-14 11:28:36,ATM Withdrawal,500,MER2072,2025-02
RT_TXN20008,ACC1056,2025-01-30 12:26:36,Withdrawal,150000,MER2043,2025-01
RT_TXN20009,ACC1087,2025-02-08 03:23:36,Deposit,50000,MER2066,2025-02
